In [1]:
import config as cfg
from data_acquirer import get_data, gen_data
from data_processor import process
import helpers
import heuristic
from heuristic import Path
from solver import solve_pdpcr

import copy
import datetime
from docplex.mp.model import Model
import logging
import heapq
import numpy as np
import time

logging.basicConfig(format=cfg.LOG_FORMATTER, level=cfg.LOG_LEVEL, datefmt='%d-%b-%y %H:%M:%S')

In [ ]:
method = 5
data_name = 'generated3.csv' #real data

if False:
    is_depot_task = True
else:
    is_depot_task = False #TODO remove this

    raw = get_data(data_name,'csv', is_depot_task)

In [ ]:

#WR2 data
sec_names = ['100','200','400','600','800','1000']
third_names = ['1','2','3','4','5']


data_paths =[f'wr2/bar-n{j}-{k}-3.csv'for j in sec_names for k in third_names]#TODO add data
dist_paths = [f'wr2/bar-n{j}-{k}-dist.csv'for j in sec_names for k in third_names]#TODO add data
num_robots = [5*i for i in range(2,3)]

method = 5
file_type = 'csv'
is_depot_task = False


In [2]:

#Li larger data

first_names = ['LC1']
sec_names = [2,4,6,8,10]
third_names = [i for i in range(1,11)]
data_paths =[f'lim/{j}00/{i}_{j}_{k}-5-uniform.csv'for i in first_names for j in sec_names for k in third_names]#
num_robots = [10*i for i in range(1,4)]
method = 5
file_type = 'csv'
is_depot_task = False

In [ ]:
num_robots = np.random.randint(1,[2,3,4,5,6,7,8,9,10,11],[10,10])
num_robots

In [ ]:
num_robots_fixed = np.array([[ 1,  2,  3,  1,  1,  2,  3,  3,  2,  9],
       [ 1,  1,  1,  1,  1,  1,  3,  4,  8,  8],
       [ 1,  2,  2,  1,  2,  3,  5,  4,  4, 10],
       [ 1,  1,  3,  4,  1,  2,  7,  1,  6,  6],
       [ 1,  1,  1,  3,  5,  2,  3,  6,  7,  7],
       [ 1,  1,  3,  2,  1,  5,  5,  4,  4,  9],
       [ 1,  2,  3,  2,  4,  1,  4,  7,  7,  6],
       [ 1,  1,  3,  4,  3,  6,  3,  2,  8,  4],
       [ 1,  2,  1,  1,  5,  3,  3,  4,  4,  9],
       [ 1,  2,  2,  3,  4,  4,  7,  5,  9,  7]])

num_robots_fixed2 = np.array([[1, 1, 2, 1, 5, 1, 7, 7, 4, 3],
       [1, 2, 1, 3, 5, 3, 3, 2, 4, 4],
       [1, 1, 2, 1, 4, 1, 4, 4, 1, 7],
       [1, 2, 3, 3, 2, 5, 5, 8, 9, 8],
       [1, 1, 3, 4, 4, 6, 6, 3, 6, 4],
       [1, 1, 1, 4, 1, 2, 5, 2, 5, 5],
       [1, 1, 2, 1, 5, 6, 1, 6, 4, 1],
       [1, 1, 1, 4, 2, 2, 3, 5, 6, 5],
       [1, 2, 1, 4, 4, 2, 2, 4, 4, 6],
       [1, 2, 1, 3, 1, 2, 5, 8, 9, 4]])

In [3]:
# CPLEX data generated
# data_idcs = [4,5,16,18]
# data_paths = [f'generated{i}.csv' for i in data_idcs]
num_robots = [10]
file_type = 'csv'
i = 0
is_depot_task = False
for data_path in data_paths:
    for num_robot in num_robots:
        for i in range(10):
            try:
                # cur_num_robots = []
                # cur_num_robots = num_robots_fixed2[:,i]
        

                raw = get_data(data_path, file_type, is_depot_task)
                avg_num_robots = np.mean([raw.tasks[i].n for i in range(len(raw.tasks))])
                print(f'Solving for {data_path} with {num_robot} robots with average n_j = {avg_num_robots}')
                method = 6
                processed_data_flow = process(raw, method)
                solve_result_flow = solve_pdpcr(processed_data_flow, num_robot, method)
                # method = 2
                # processed_data_flow_old = process(raw, method)
                # solve_result_flow_old = solve_pdpcr(processed_data_flow_old, num_robot, method)
                

                solve_log = []
                solve_log.append(data_path)
                solve_log.append(num_robot)

                # solve_log.append(solve_result_flow_old['solve_time'])
                # solve_log.append(solve_result_flow_old['sol'].objective_value)
                # solve_log.append(solve_result_flow_old['processed_nodes'])
                # solve_log.append(solve_result_flow_old['gap'])

                solve_log.append(solve_result_flow['solve_time'])
                solve_log.append(solve_result_flow['sol'].objective_value)
                solve_log.append(solve_result_flow['processed_nodes'])
                solve_log.append(solve_result_flow['gap'])

                now = datetime.datetime.now()
                solve_log.append(avg_num_robots)
                solve_log.append(now.strftime("%Y-%m-%d %H:%M:%S"))
                with open('solve_results/solve_log_li_lim_flow_6.csv', 'a') as f:
                    f.write(','.join([str(i) for i in solve_log])+'\n')
            except Exception as e:
                print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
                print(e)
            i += 1

Solving for lim/200/LC1_2_1-5-uniform.csv with 10 robots with average n_j = 3.018867924528302
C:\Users\CTNGUY~1\AppData\Local\Temp\PDPCR_Flow_Based_Makespan.lp
Version identifier: 22.1.0.0 | 2022-03-09 | 1a383f8ce
CPXPARAM_Read_DataCheck                          1
CPXPARAM_TimeLimit                               1800
CPXPARAM_MIP_Tolerances_MIPGap                   0
Tried aggregator 2 times.
MIP Presolve eliminated 3851 rows and 215 columns.
MIP Presolve modified 33290 coefficients.
Aggregator did 3530 substitutions.
Reduced MIP has 27502 rows, 19475 columns, and 88708 nonzeros.
Reduced MIP has 11448 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.11 sec. (103.93 ticks)
Probing time = 0.08 sec. (10.97 ticks)
Tried aggregator 1 time.
Detecting symmetries...
Reduced MIP has 27502 rows, 19475 columns, and 88708 nonzeros.
Reduced MIP has 11448 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.06 sec. (55.09 ticks)
Probing time = 0.09 sec. (10.95 ticks)
Cl

: 

In [ ]:
# read file 'solve_results/solve_log_generated_progressive_large_nj.csv' to pandas dataframe, it has the following columns:
# data_name, total_robots, solve_time_vrp, obj_value_vrp, processed_nodes_vrp, gap_vrp, solve_time_flow, obj_value_flow, processed_nodes_flow, gap_flow, avg_num_robots,date
import pandas as pd
df_large_nj = pd.read_csv('solve_results/solve_log_generated_progressive_large_nj.csv', header=None)
df_large_nj.columns = ['data_name', 'total_robots', 'solve_time_vrp', 'obj_value_vrp', 'processed_nodes_vrp', 'gap_vrp', 'solve_time_flow', 'obj_value_flow', 'processed_nodes_flow', 'gap_flow','avg_num_robots', 'date']


In [ ]:
#plot data, y column corresponding to solve_time, x column corresponding to avg_num_robots for vrp and flow
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
first_names = ['lc','lr','lrc']
sec_names = [i for i in range(101,115)] +[i for i in range(201,215)]
# third_names = [i for i in range(1,11)]
data_paths =[f'lim/100/{i}{j}-5-uniform.csv'for i in first_names for j in sec_names]#
num_robots = [10*i for i in range(1,2)]
file_type = 'csv'
is_depot_task = False

In [3]:
## Mass solve using tabu_search
num_robots = [10,20,30]
for num_robot in num_robots:
    for data_path in data_paths:
        try:
            logging.info
            raw = get_data(data_path, file_type, is_depot_task)
            # processed_data = process(raw, method)
            processed_h = heuristic.process_greedy_1(raw)
            depot = processed_h.depot
            tasks = processed_h.tasks
            cfg.TABU_MAX_TL_ADD_SIZE = int(len(tasks)/1.2)
            cfg.TABU_MAX_TL_DROP_SIZE = int(len(tasks)/1.2)
            c_travel = helpers.create_task_dist_dict(depot,tasks)
            c_operate = helpers.create_operate_dict(depot,tasks)

            tik1 = time.time()
            tabu = heuristic.TabuSearch(processed_h,c_travel, c_operate)
            best_sol, best_sol_obj = tabu.search(num_robot,cfg.TABU_MAX_ITER)
            tok1 = time.time()
            tabu_time = tok1 - tik1
            print(f'Tabu time: {tabu_time}')
        
            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(best_sol_obj)
            solve_log.append(np.round(tabu_time,2))
            with open('solve_log_tabu.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)

08-Jun-24 20:57:53:INFO:TABU SEARCH PARAMETERS
08-Jun-24 20:57:53:INFO:RANDOM_SEED: 152
08-Jun-24 20:57:53:INFO:NUM_REVERSE_NEIGHBORS: 25
08-Jun-24 20:57:53:INFO:NUM_REROUTE_NEIGHBORS: 50
08-Jun-24 20:57:53:INFO:MAX_TL_ADD_SIZE: 88
08-Jun-24 20:57:53:INFO:MAX_TL_DROP_SIZE: 88
08-Jun-24 20:57:53:INFO:MAX_ITER: 5000


Solving for lim/200/LC1_2_1-5-uniform.csv with 10 robots
Viet generalized sol is 973.69
Viet generalized sol is 1019.69
Viet generalized sol is 1048.17
Viet generalized sol is 966.5
Viet generalized sol is 917.01
Viet generalized sol is 931.97
Viet generalized sol is 987.45
Viet generalized sol is 1009.15
Viet generalized sol is 1005.74
Viet generalized sol is 975.19
Viet generalized sol is 916.48
Viet generalized sol is 1056.15
Viet generalized sol is 1083.75
Viet generalized sol is 940.29
Viet generalized sol is 941.82
Viet generalized sol is 933.03
Viet generalized sol is 1084.55
Viet generalized sol is 1048.35
Viet generalized sol is 1028.88
Viet generalized sol is 931.45
Viet generalized sol is 962.06
Viet generalized sol is 1048.3
Viet generalized sol is 1033.72
Viet generalized sol is 1044.07
Viet generalized sol is 1024.97
Viet generalized sol is 952.63
Viet generalized sol is 1048.63


08-Jun-24 20:57:56:INFO:Initial makespan: 916.484321567792


Viet generalized sol is 966.52
Viet generalized sol is 926.94
Viet generalized sol is 1055.24


08-Jun-24 20:57:56:INFO:Best fitness updated: 913.0349250541378
08-Jun-24 20:57:56:INFO:Iteration 0 done
08-Jun-24 20:57:57:INFO:Best fitness updated: 910.458145763421
08-Jun-24 20:57:57:INFO:Best fitness updated: 909.5226732843391
08-Jun-24 20:57:57:INFO:Best fitness updated: 909.1641764443345
08-Jun-24 20:57:58:INFO:Best fitness updated: 908.4048585470284
08-Jun-24 20:57:58:INFO:Best fitness updated: 904.8208326010586
08-Jun-24 20:57:59:INFO:Best fitness updated: 904.7709465859184
08-Jun-24 20:58:00:INFO:Best fitness updated: 904.3224893231959
08-Jun-24 20:58:08:INFO:Iteration 50 done
08-Jun-24 20:58:25:INFO:Iteration 100 done
08-Jun-24 20:58:37:INFO:Iteration 150 done
08-Jun-24 20:58:48:INFO:Iteration 200 done
08-Jun-24 20:58:59:INFO:Iteration 250 done
08-Jun-24 20:59:10:INFO:Iteration 300 done
08-Jun-24 20:59:21:INFO:Iteration 350 done
08-Jun-24 20:59:32:INFO:Iteration 400 done
08-Jun-24 20:59:43:INFO:Iteration 450 done
08-Jun-24 20:59:55:INFO:Iteration 500 done
08-Jun-24 20:59:59:

Tabu time: 1111.2090656757355
Solving for lim/200/LC1_2_2-5-uniform.csv with 10 robots
Viet generalized sol is 1271.94
Viet generalized sol is 1310.49
Viet generalized sol is 1178.86
Viet generalized sol is 1276.33
Viet generalized sol is 1132.18
Viet generalized sol is 1230.5
Viet generalized sol is 1295.57
Viet generalized sol is 1093.55
Viet generalized sol is 1205.82
Viet generalized sol is 1198.9
Viet generalized sol is 1231.49
Viet generalized sol is 1214.2
Viet generalized sol is 1220.29
Viet generalized sol is 1252.76
Viet generalized sol is 1386.23
Viet generalized sol is 1394.64
Viet generalized sol is 1331.55
Viet generalized sol is 1305.31
Viet generalized sol is 1497.39
Viet generalized sol is 1065.49
Viet generalized sol is 1414.86
Viet generalized sol is 1370.35
Viet generalized sol is 1221.07
Viet generalized sol is 1313.82
Viet generalized sol is 1213.15
Viet generalized sol is 1226.38
Viet generalized sol is 1279.2
Viet generalized sol is 1387.16
Viet generalized sol 

... (521 outputs left out)

Viet generalized sol is 3437.66
Viet generalized sol is 3681.69
Viet generalized sol is 3501.68
Viet generalized sol is 3760.19
Viet generalized sol is 3530.75
Viet generalized sol is 3557.25
Viet generalized sol is 3445.69
Viet generalized sol is 3590.58
Viet generalized sol is 3591.52
Viet generalized sol is 3401.88
Viet generalized sol is 3606.63
Viet generalized sol is 3593.48
Viet generalized sol is 3559.51
Viet generalized sol is 3420.19
Viet generalized sol is 3550.03
Viet generalized sol is 3474.55
Viet generalized sol is 3733.25
Viet generalized sol is 3533.19
Viet generalized sol is 3575.68
Viet generalized sol is 3546.82
Viet generalized sol is 3810.65
Viet generalized sol is 3606.34
Viet generalized sol is 3668.64
Viet generalized sol is 3330.52
Viet generalized sol is 3627.67
Viet generalized sol is 3776.84
Viet generalized sol is 3688.04
Viet generalized sol is 3554.5
Viet generalized sol is 3387.38


11-Jun-24 17:53:15:INFO:Initial makespan: 3330.5231513287654


Viet generalized sol is 3806.31


11-Jun-24 17:53:16:INFO:Best fitness updated: 3329.9630184484554
11-Jun-24 17:53:16:INFO:Iteration 0 done
11-Jun-24 17:53:22:INFO:Best fitness updated: 3328.398029188347
11-Jun-24 17:53:25:INFO:Best fitness updated: 3324.983185344916
11-Jun-24 17:53:26:INFO:Best fitness updated: 3324.412642564006
11-Jun-24 17:53:27:INFO:Best fitness updated: 3323.824136822872
11-Jun-24 17:53:34:INFO:Best fitness updated: 3323.0160291046677
11-Jun-24 17:53:38:INFO:Best fitness updated: 3322.9823064478414
11-Jun-24 17:53:39:INFO:Best fitness updated: 3321.7796994935024
11-Jun-24 17:53:44:INFO:Best fitness updated: 3321.4945046020985
11-Jun-24 17:53:49:INFO:Best fitness updated: 3319.755991537536
11-Jun-24 17:53:53:INFO:Best fitness updated: 3318.972869607102
11-Jun-24 17:54:05:INFO:Iteration 50 done
11-Jun-24 17:54:32:INFO:Best fitness updated: 3318.9376811081684
11-Jun-24 17:54:35:INFO:Best fitness updated: 3318.9277205610997
11-Jun-24 17:54:36:INFO:Best fitness updated: 3318.5860437987617
11-Jun-24 17:

In [ ]:
## mass solve using heuristics
for data_path in data_paths:
    for num_robot in num_robots:
        try:
            print(f'Solving for {data_path} with {num_robot} robots')
            raw = get_data(data_path, file_type, is_depot_task)
            processed_data = process(raw, method)
            processed_h = heuristic.process_greedy_1(raw)
            tik1 = time.time()
            solve_result_h = heuristic.solve_greedy_1(processed_h, num_robot)
            tok1 = time.time()
            greedy1_time = tok1 - tik1
            print(f'Greedy 1 time: {greedy1_time}')
            tik2 = time.time()
            solve_result_h_jp = heuristic.solve_greedy_jp(processed_h, num_robot)
            tok2 = time.time()
            greedy_jp_time = tok2 -tik2
            print(f'Greedy jp time: {greedy_jp_time}')
            tik3 = time.time()
            solve_result_h_jp_2 = heuristic.solve_greedy_jp_compare_start_time(processed_h, num_robot)
            tok3 = time.time()
            greedy_jp_time_2 = tok3 -tik3
            print(f'Greedy jp time 2: {greedy_jp_time_2}')

        
            solve_log = []
            solve_log.append(data_path)
            solve_log.append(solve_result_h.get_makespan().round(2))
            solve_log.append(np.round(greedy1_time,2))
            solve_log.append(solve_result_h_jp.get_makespan().round(2))
            solve_log.append(np.round(greedy_jp_time,2))
            solve_log.append(solve_result_h_jp_2.get_makespan().round(2))
            solve_log.append(np.round(greedy_jp_time_2,2))
            with open('solve_log_h_2.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)

In [ ]:
import helpers
## Mass solve 2 opt
for data_path in data_paths:
    for num_robot in num_robots:
        try:
            print(f'Solving for {data_path} with {num_robot} robots')
            raw = get_data(data_path, file_type, is_depot_task)
            processed_h = heuristic.process_greedy_1(raw)

            tik2 = time.time()
            solve_result_h2 = heuristic.solve_greedy_jp(processed_h, num_robot)
            tok2 = time.time()
            greedy_h2 = tok2 -tik2

            tik3 = time.time()
            solve_result_h3 = heuristic.solve_greedy_jp_compare_start_time(processed_h, num_robot)
            tok3 = time.time()
            greedy_jp_time_2 = tok3 -tik3

            depot = processed_h.depot
            tasks = processed_h.tasks
            c_travel = helpers.create_task_dist_dict(depot,tasks)
            c_operate = helpers.create_operate_dict(depot,tasks)
            tik = time.time()
            solve_opt = heuristic.TwoOptSolver(processed_h,c_travel, c_operate)
            graph,makespan_2opt = solve_opt.solve(total_robots= num_robot)
            tok = time.time()
            two_opt_time = tok - tik

            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(solve_result_h2.get_makespan().round(2))
            solve_log.append(np.round(greedy_h2,2))
            solve_log.append(solve_result_h3.get_makespan().round(2))
            solve_log.append(np.round(greedy_jp_time_2,2))
            solve_log.append(makespan_2opt.round(2))
            solve_log.append(np.round(two_opt_time,2))
            now = datetime.datetime.now()
            solve_log.append(now.strftime("%Y-%m-%d %H:%M:%S"))
            with open('solve_log_h_2opt.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)

In [ ]:
## Mass solve Viet 

import logging
logging.basicConfig(format=cfg.LOG_FORMATTER, level=cfg.LOG_LEVEL, datefmt='%d-%b-%y %H:%M:%S')
consider_process_time =True
for data_path in data_paths:
    for num_robot in num_robots:
        try:
            print(f'Solving for {data_path} with {num_robot} robots')
            raw = get_data(data_path, file_type, is_depot_task)
            processed_h = heuristic.process_greedy_1(raw)
            depot = processed_h.depot
            tasks = processed_h.tasks
            c_travel = helpers.create_task_dist_dict(depot,tasks)
            c_operate = helpers.create_operate_dict(depot,tasks)
            tik = time.time()
            solve_viet = heuristic.VietSolverOrig(processed_h, c_travel, c_operate)
            solve_viet.solve(num_robot,consider_process_time)
            tok = time.time()
            time_viet = tok - tik
            print(f'makespan: {solve_viet.get_makespan()}')
            print('Done')

            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(solve_viet.get_makespan().round(2))
            solve_log.append(np.round(time_viet,2))
            with open('solve_log_h_Viet_09-04-process-time.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)


In [ ]:
## Mass solve Viet k tasks

import logging
logging.basicConfig(format=cfg.LOG_FORMATTER, level=cfg.LOG_LEVEL, datefmt='%d-%b-%y %H:%M:%S')

tasks_each_iters = [5]
consider_process_time =False
for task_each_ite in tasks_each_iters:
    for data_path in data_paths:
        for num_robot in num_robots:
            try:
                print(f'Solving for {data_path} with {num_robot} robots')
                raw = get_data(data_path, file_type, is_depot_task)
                processed_h = heuristic.process_greedy_1(raw)
                depot = processed_h.depot
                tasks = processed_h.tasks
                c_travel = helpers.create_task_dist_dict(depot,tasks)
                c_operate = helpers.create_operate_dict(depot,tasks)
                tik = time.time()
                solve_viet = heuristic.VGeneralizedSolver(processed_h, c_travel, c_operate)
                sol = solve_viet.solve(num_robot,task_each_ite,consider_process_time)
                tok = time.time()
                time_viet = tok - tik
                makespan = sol.get_makespan()
                print(f'makespan: {makespan}')
                print('Done')

                solve_log = []
                solve_log.append(data_path)
                solve_log.append(num_robot)
                solve_log.append(task_each_ite)
                solve_log.append(makespan.round(2))
                solve_log.append(np.round(time_viet,2))
                with open('solve_log_h_Viet k task large instances 23-04.csv', 'a') as f:
                    f.write(','.join([str(i) for i in solve_log])+'\n')
            except Exception as e:
                print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
                print(e)


In [1]:
a ={0: 0.0, 1: 599.9817763899168, 2: 271.90904527065663, 3: 283.1551518161024, 4: 142.61641735789044, 5: 239.53321272842314, 6: 503.8917411508151, 7: 320.3896980241406, 8: 347.00099826945745, 9: 177.80785134520917, 10: 194.39276118209756, -1: 0}

In [3]:
for i in a:
    print(i)

0
1
2
3
4
5
6
7
8
9
10
-1


In [ ]:
## Mass solve Viet k tasks with random k

import logging
logging.basicConfig(format=cfg.LOG_FORMATTER, level=cfg.LOG_LEVEL, datefmt='%d-%b-%y %H:%M:%S')

ite = 100
tasks_each_iter_ranges = [[1,3],[1,4],[1,5]]
consider_process_time =False
for task_each_ite in tasks_each_iter_ranges:
    for data_path in data_paths:
        for num_robot in num_robots:
            try:
                print(f'Solving for {data_path} with {num_robot} robots')
                raw = get_data(data_path, file_type, is_depot_task)
                processed_h = heuristic.process_greedy_1(raw)
                depot = processed_h.depot
                tasks = processed_h.tasks
                c_travel = helpers.create_task_dist_dict(depot,tasks)
                c_operate = helpers.create_operate_dict(depot,tasks)

                solve_result_h_jp = heuristic.solve_greedy_jp_compare_start_time(processed_h, num_robot)
                TT_makespan = solve_result_h_jp.get_makespan()

                tik = time.time()
                cur_best = np.infty
                cur_worst = -np.infty
                makespan_results = []
                no_better_TT = 0

                for i in range(ite):
                    solve_viet = heuristic.VGeneralizedSolver(processed_h, c_travel, c_operate)
                    sol = solve_viet.solve(num_robot,'random',consider_process_time,task_range=task_each_ite)
                    makespan = sol.get_makespan()
                    makespan_results.append(makespan)
                    if makespan < cur_best:
                        cur_best = makespan
                    if makespan > cur_worst:
                        cur_worst = makespan
                    if makespan <= TT_makespan:
                        no_better_TT +=1
                avg_makespan = np.mean(makespan_results)
                tok = time.time()
                time_viet = tok - tik
            
                solve_log = []
                solve_log.append(data_path)
                solve_log.append(num_robot)
                solve_log.append(str(f'{task_each_ite[0]}-{task_each_ite[1]}'))
                solve_log.append(cur_best.round(2))
                solve_log.append(cur_worst.round(2))
                solve_log.append(avg_makespan.round(2))
                solve_log.append(np.round(time_viet,2))
                solve_log.append(no_better_TT)
                with open('solve_log_h_Viet k task random large instances 29-04.csv', 'a') as f:
                    f.write(','.join([str(i) for i in solve_log])+'\n')
            except Exception as e:
                print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
                print(e)


In [ ]:
raw = get_data(data_path, file_type, is_depot_task)
processed_h = heuristic.process_greedy_1(raw)
depot = processed_h.depot
tasks = processed_h.tasks
c_travel = helpers.create_task_dist_dict(depot,tasks)
c_operate = helpers.create_operate_dict(depot,tasks)
tik = time.time()
solve_opt = heuristic.TwoOptSolver(processed_h,c_travel, c_operate)

In [ ]:
#Get data

#Read file solve_log.csv, save it as a dataframe
import pandas as pd
solve_cplex_df = pd.read_csv('solve_log.csv', header=None)
solve_cplex_df.columns = ['data_path', 'num_robot', 'solve_time', 'objective_value', 'processed_nodes', 'gap', 'solve_date']

solve_h_df = pd.read_csv('solve_log_h.csv', header=None)
solve_h_df.columns = ['data_path', 'objective_value_1', 'time_greedy_1', 'objective_value_jp', 'time_greedy_jp']

In [ ]:
# right join 2 dataframes on column data_path
merged_df = pd.merge(solve_cplex_df, solve_h_df, on='data_path', how='right')

In [ ]:
merged_df['diff_h_1'] = np.round(1- merged_df['objective_value_1']/merged_df['objective_value'],2)
merged_df['diff_h_jp'] = np.round(1- merged_df['objective_value_jp']/merged_df['objective_value'],2)
#change order of columns in meged_df as follow: data_path, num_robot, objective_value, solve_time, objective_value_1, time_greedy_1, diff_h_1, objective_value_jp, time_greedy_jp, diff_h_jp,
merged_df = merged_df[['data_path', 'num_robot', 'objective_value', 'solve_time', 'objective_value_1', 'time_greedy_1', 'diff_h_1', 'objective_value_jp', 'time_greedy_jp', 'diff_h_jp']]
merged_df
#Remove the 'lim' prefix from data_path
merged_df['data_path'] = merged_df['data_path'].str.replace('lim/','')
#round solve_time to 2 decimal places
merged_df['solve_time'] = np.round(merged_df['solve_time'],2)
#round objective_value to 2 decimal places
merged_df['objective_value'] = np.round(merged_df['objective_value'],2)

In [ ]:
merged_df.to_latex('merged_df.tex', index=True)

In [ ]:
#Temp
import pandas as pd
solve_h_df = pd.read_csv('solve_log_h_2.csv', header=None)
solve_h_df.columns = ['data_path', 'objective_value_1', 'time_greedy_1', 'objective_value_2', 'time_greedy_2', 'objective_value_3', 'time_greedy_3']
#Keep the following columns: data_path, objective_value_2, time_greedy_2
solve_h_df = solve_h_df[['data_path', 'objective_value_2', 'time_greedy_2']]

In [ ]:
solve_h_df_2opt = pd.read_csv('solve_log_h_2opt.csv', header=None)
solve_h_df_2opt.columns = ['data_path', 'num_robot', 'objective_value_3', 'time_greedy_3', 'objective_value_2opt', 'time_2opt']

In [ ]:
#Join 2 dataframes on data_path
merged_df_2 = pd.merge(solve_h_df, solve_h_df_2opt, on='data_path', how='right')
#Change the order of columns as follow: data_path, num_robot, objective_value_2, time_greedy_2, objective_value_3, time_greedy_3, objective_value_2opt, time_2opt
merged_df_2 = merged_df_2[['data_path', 'num_robot', 'objective_value_2', 'time_greedy_2', 'objective_value_3', 'time_greedy_3', 'objective_value_2opt', 'time_2opt']]

In [ ]:
#save the data as a latex table
merged_df_2.to_latex('merged_df_2.tex', index=True)